# Road Damage Detection with YOLO11 — Potholes, Cracks & Manholes
Fine-tuning **YOLO11n** (Ultralytics) on the Kaggle *Road Damage Dataset* by Lorenzo Arcioni.
Classes: `0 Pothole`, `1 Crack`, `2 Manhole`.

## 1. Download the dataset with kagglehub

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes")

print("Path to dataset files:", path)

/workspace/venv3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/box/.cache/kagglehub/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes/versions/4


## 2. Inspect the dataset
`labels/` holds 4-point polygons; `labels-YOLO/` holds the same objects as axis-aligned YOLO boxes
(`class x_center y_center width height`, normalised). We train on the YOLO boxes.

In [2]:
import os, glob, random, shutil, collections
from pathlib import Path

DATA = Path(path) / "data"
CLASSES = ["Pothole", "Crack", "Manhole"]
images = sorted(glob.glob(str(DATA / "images" / "*.jpg")))
print("Images:", len(images))
counts = collections.Counter()
for lbl in glob.glob(str(DATA / "labels-YOLO" / "*.txt")):
    for line in open(lbl):
        if line.strip():
            counts[CLASSES[int(line.split()[0])]] += 1
print("Objects per class:", dict(counts))

Images: 2009


Objects per class: {'Manhole': 957, 'Pothole': 1261, 'Crack': 2519}


## 3. Build a YOLO train/val split (80/20, seed 42)

In [3]:
WORK = Path("road_damage_yolo")
random.seed(42)
shuffled = images[:]
random.shuffle(shuffled)
n_val = int(0.2 * len(shuffled))
splits = {"val": shuffled[:n_val], "train": shuffled[n_val:]}
for split, files in splits.items():
    (WORK / "images" / split).mkdir(parents=True, exist_ok=True)
    (WORK / "labels" / split).mkdir(parents=True, exist_ok=True)
    for img in files:
        stem = Path(img).stem
        shutil.copy(img, WORK / "images" / split / Path(img).name)
        src = DATA / "labels-YOLO" / f"{stem}.txt"
        dst = WORK / "labels" / split / f"{stem}.txt"
        if src.exists():
            shutil.copy(src, dst)
        else:
            open(dst, "w").close()
    print(split, len(files))

yaml_path = WORK / "data.yaml"
yaml_path.write_text(
    f"path: {WORK.resolve()}\ntrain: images/train\nval: images/val\n"
    "names:\n  0: Pothole\n  1: Crack\n  2: Manhole\n"
)
print(yaml_path.read_text())

val 401


train 1608
path: /workspace/p3run/road_damage_yolo
train: images/train
val: images/val
names:
  0: Pothole
  1: Crack
  2: Manhole



## 4. Fine-tune YOLO11n

In [4]:
from ultralytics import YOLO

RUNS = Path("runs").resolve()  # absolute, so outputs land next to the notebook
EPOCHS = int(os.environ.get("EPOCHS", 30))
model = YOLO("yolo11n.pt")
model.train(data=str(yaml_path), epochs=EPOCHS, imgsz=640, batch=16, workers=4,
            seed=42, project=str(RUNS), name="road_damage", exist_ok=True, plots=True)

Ultralytics 8.4.194 🚀 Python-3.13.5 torch-2.15.0+cpu CPU (Intel Xeon Processor)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=road_damage_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=road_damage, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, 

Overriding model.yaml nc=80 with nc=3



                   from  n    params  module                                       arguments                     


  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 


  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                


... (3,573 outputs left out)

               Manhole        162        195      0.609      0.851       0.77      0.398


Speed: 0.2ms preprocess, 6.1ms inference, 0.0ms loss, 0.3ms postprocess per image


Results saved to /workspace/p3run/runs/road_damage


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f2ea5039a20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

## 5. Evaluate the best weights on the validation set

In [5]:
best = YOLO(RUNS / "road_damage" / "weights" / "best.pt")
metrics = best.val(data=str(yaml_path), imgsz=640, split="val", plots=True,
                   project=str(RUNS), name="road_damage_val", exist_ok=True)
print(f"mAP50     : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
for i, name in enumerate(CLASSES):
    p, r, ap50, ap = metrics.box.class_result(i)
    print(f"{name:8s} P={p:.3f} R={r:.3f} mAP50={ap50:.3f} mAP50-95={ap:.3f}")

Ultralytics 8.4.194 🚀 Python-3.13.5 torch-2.15.0+cpu CPU (Intel Xeon Processor)


YOLO11n summary (fused): 101 layers, 2,597,537 parameters, 0 gradients, 6.4 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2480.2±1156.8 MB/s, size: 116.5 KB)


val: Scanning /workspace/p3run/road_damage_yolo/labels/val.cache... 401 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 401/401 120.1Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 3% ──────────── 1/26 1.7it/s 0.2s<14.6s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 7% ╸─────────── 2/26 2.8it/s 0.4s<8.6s

... (27 outputs left out)

               Manhole        162        195      0.609      0.851       0.77      0.398


Speed: 0.2ms preprocess, 5.8ms inference, 0.0ms loss, 0.4ms postprocess per image


Results saved to /workspace/p3run/runs/road_damage_val


mAP50     : 0.4964
mAP50-95  : 0.2247
Precision : 0.5308
Recall    : 0.5000
Pothole  P=0.574 R=0.283 mAP50=0.383 mAP50-95=0.148
Crack    P=0.410 R=0.366 mAP50=0.336 mAP50-95=0.128
Manhole  P=0.609 R=0.851 mAP50=0.770 mAP50-95=0.398


## 6. Predict on validation images

In [6]:
os.makedirs("predictions", exist_ok=True)
samples = sorted(splits["val"])[:8]
for img in samples:
    res = best.predict(img, imgsz=640, conf=0.25, verbose=False)[0]
    found = collections.Counter(CLASSES[int(c)] for c in res.boxes.cls)
    print(Path(img).name, dict(found))
    res.save(filename=f"predictions/{Path(img).stem}_pred.jpg")

20250219_164813.jpg {'Manhole': 1, 'Crack': 1, 'Pothole': 1}
20250219_164912.jpg {'Pothole': 1}
20250219_164919.jpg {'Pothole': 1}
20250219_165100.jpg {'Pothole': 2}


20250219_165333.jpg {'Pothole': 3}
20250219_165411.jpg {'Pothole': 4}
20250219_170253.jpg {'Crack': 1}
20250219_170332.jpg {'Crack': 1}


## 7. Export to ONNX for in-browser inference

In [7]:
onnx_path = best.export(format="onnx", imgsz=640, opset=12, simplify=True)
print("ONNX model:", onnx_path)

Ultralytics 8.4.194 🚀 Python-3.13.5 torch-2.15.0+cpu CPU (Intel Xeon Processor)


💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino


YOLO11n summary (fused): 101 layers, 2,597,537 parameters, 0 gradients, 6.4 GFLOPs



PyTorch: starting from '/workspace/p3run/runs/road_damage/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (5.2 MB)



ONNX: starting export with onnx 1.23.2 opset 12...


ONNX: slimming with onnxslim 0.1.98...


ONNX: export success ✅ 1.2s, saved as '/workspace/p3run/runs/road_damage/weights/best.onnx' (10.2 MB)



Export complete (1.3s)
Results saved to /workspace/p3run/runs/road_damage/weights/best.onnx
Predict:         yolo predict task=detect model=/workspace/p3run/runs/road_damage/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/workspace/p3run/runs/road_damage/weights/best.onnx imgsz=640 data=road_damage_yolo/data.yaml 
Visualize:       https://netron.app


ONNX model: /workspace/p3run/runs/road_damage/weights/best.onnx
